In [ ]:
import numpy as np
import cv2
from matplotlib import pyplot as plt

## Lab 2: Image Stitching with Homographies

In this lab, we will explore image stitching and panorama creation. You will use a homography to register two images together, and implement bilinear interpolation to help accomplish this. As part of the lab, you will also explore image enhancement strategies to improve the quality of image stitching at the seam (the join between two images) relying on your knowledge of image brightness and human perception of this. Collectively, tasks 1-5 will provide a final stitched image. In task 6, you will apply the same code written in tasks 1-5 to create a panorama image using your own photos.

* Task 1: Draw test points on the left image
* Task 2: Use a homography to find the location of these points in the right image
* Task 3: Bilinear interpolation of the right image pixels
* Task 4: Image stitching 
* Task 5: Better blending
* Task 6: Now try your own!

### Academic integrity

Every lab submission will be screened for any collusion and/or plagiarism. Breaches of academic integrity will be investigated thoroughly and may result in a zero for the assessment along with interviews with the plagiarism officers at Monash University.

### Late submissions

The default late submission university penalty will apply.

### Lab Instructions and the Use of Generative AI

You may not use any built-in opencv functions for this lab, other than those used for loading/ saving an image, extracting and matching keypoints, and
computing homographies.
* You may use NumPy for array handling, and vectorizing your code (reducing
the number of for-loops) is encouraged.
* You should use Matplotlib to display images and any intermediate results.
* You may use generative AI.

### Grading
Each lab is worth 8%, and there are a number of sections and tasks with their own weighting. A task is only considered complete if you can demonstrate a working program and show an understanding of the underlying concepts. Note that later tasks should reuse code from earlier tasks.

Masks will be provided based on the correctness of the code, the quality of your results, comments indicating you understand your work, and a discussion of tasks provided at the end of the notebook. 

# Task 1: Draw test points on the left image

Draw the following points on the left image as red crosses. Display the resulting image.

{446 , 499, 1}, {383, 590, 1}, {296 , 499, 1}, {282, 511, 1}, {401 , 508, 1}

Recall from lectures that these 3-element homogeneous coordinates can be transformed to 2D image pixel coordinates by dividing the first and second elements by the third (needed for later tasks).

In [ ]:
# Write your code here
img_left = cv2.imread('left.jpg')
x1, y1 = 446, 499
x2, y2 = 383, 590
x3, y3 = 296, 499
x4, y4 = 282, 511
x5, y5 = 401, 508


In [ ]:
# Show results here
plt.imshow(img_left)
plt.plot(x1, y1, 'rx')
plt.plot(x2, y2, 'rx')
plt.plot(x3, y3, 'rx')
plt.plot(x4, y4, 'rx')
plt.plot(x5, y5, 'rx')
plt.show()

# Task 2: Use Homography to find right image points


The following homography transforms pixel coordinates between the left and right images as
$$ x_R = H x_L $$

\begin{bmatrix}
1.6011 & 0.0277 & -393.5701 \\
0.3242 & 1.5119 & -228.8918 \\
0.0009 & 0.0002 & 1.0000
\end{bmatrix}

Apply the homography to transform the left image points in Task 1 to their corresponding locations in the right image. Draw the transformed points as red crosses on the right image. Check your result before moving on.


In [ ]:
# Write your code here
homography = np.array([[1.6011, 0.0277, -393.5701], [0.3242, 1.5119, -228.8918], [0.0009, 0.0002, 1.0000]])
img_points = np.array([[x1, y1], [x2, y2], [x3, y3], [x4, y4], [x5, y5]])

# apply homography to the image points to get
img_points_homogeneous = np.hstack((img_points, np.ones((img_points.shape[0], 1))))
right_points_homogeneous = np.dot(homography, img_points_homogeneous.T).T
right_points = right_points_homogeneous[:, :2] / right_points_homogeneous[:, 2:]

img_right = cv2.imread('right.jpg')
plt.figure()
plt.imshow(img_right)
plt.plot(right_points[:, 0], right_points[:, 1], 'rx')
plt.show()


In [ ]:
# Print results here
print("The coordinates of the right points are:")
for i, point in enumerate(right_points):
    print(f"Point {i+1}: ({point[0]:.2f}, {point[1]:.2f})")

print(right_points)


# Task 3: Bilinear interpolation of the right image

The transformed coordinates can be in between pixel locations. Write a bilinear interpolation function to compute the intensity of the transformed pixel coordinate in right.jpg using intensity values from neighbouring pixel locations. Print the interpolated intensity value for each transformed point in Task 2. The first point should be around 176 whereas the last point should be around 73.

HINT: The bilinear interpolation function should take the transformed pixel coordinate and the intensity values of its four neighbours as input arguments, and should output the interpolated intensity value.

In [ ]:
# Write your code here

def bilinear_interpolation(img, x, y):
    x1_point = int(np.floor(x))
    x2_point = int(np.ceil(x))
    y1_point = int(np.floor(y))
    y2_point = int(np.ceil(y))
    if x1_point < 0 or x2_point >= img.shape[1] or y1_point < 0 or y2_point >= img.shape[0]:
        return np.array([0, 0, 0])  # Return zeros for out-of-bounds
    Q11 = img[y1_point, x1_point]
    Q12 = img[y1_point, x2_point]
    Q21 = img[y2_point, x1_point]
    Q22 = img[y2_point, x2_point]
    x_diff = x - x1_point
    y_diff = y - y1_point
    R1 = (1 - x_diff) * Q11 + x_diff * Q12
    R2 = (1 - x_diff) * Q21 + x_diff * Q22
    P = (1 - y_diff) * R1 + y_diff * R2
    return P


In [ ]:
# Show results here



for i, point in enumerate(right_points):
    P = bilinear_interpolation(img_right, point[0], point[1])
    print(f"Interpolated color at Point {i+1}: {P[0]:.2f}, {P[1]:.2f}, {P[2]:.2f}")


# Task 4: Image stitching

Create a 1200x800 (width x height) image and fill the left hand side of this image with the left image. This stitched image will use the left image coordinate system (xl) throughout the stitching process. Next, fill in the remaining pixels on the RHS by transforming their pixel coordinates (left image coordinates) to the right image coordinates via the homography from Task 2 and determining the intensity using your bilinear interpolation implementation. If the right pixel  coordinate is valid, generate the pixel value using bilinear interpolation, but if the right pixel coordinate is invalid, use a pixel value of zero. Display the stitching results. It should look like a wide-angle image with a visible seam where the two images join.

In [ ]:
# Write your code here
def stitch_images(img_left, img_right, homography):
  height, width = 800, 1200
  new_img = np.zeros((height, width, 3), dtype=np.uint8)

  # fill the image with the left hand side with the original left image
  new_img[:img_left.shape[0], :img_left.shape[1]] = img_left


  # Fill the right hand side of the image:
  # Transform the left hand pixel coordinates to the right image coordinates
  # Determine the pixel value via bilinear interpolation
  for y in range(height):
      for x in range(width):
          # Transform the left hand pixel coordinates to the right image coordinates
          left_point = np.array([x, y, 1])
          right_point_homogeneous = np.dot(homography, left_point)
          right_point = right_point_homogeneous[:2] / right_point_homogeneous[2]
          #check if the bounds are within the left image for overlapping
          if (y < img_left.shape[0]) and (x < img_left.shape[1]):
              if right_point[0] < 0 or right_point[1] < 0:
                new_img[y, x] = img_left[y, x]
              else:
                # Determine the pixel value via bilinear interpolation
                new_img[y, x] = bilinear_interpolation(img_right, right_point[0], right_point[1])
          else:
              new_img[y, x] = bilinear_interpolation(img_right, right_point[0], right_point[1])
  return new_img
new_img = stitch_images(img_left, img_right, homography)

In [ ]:
# Show results here
plt.imshow(new_img)
plt.show()

# Task 5: Better blending

Improve the visual quality of the stitched image by trying the following image processing techniques:

1. Adjust the width of the output image automatically so that fewer black pixels are
visible 
2. Adjust the brightness (by a scaling factor) of each image so that the seam is less
visible
3. Apply a small amount of Gaussian blur or alpha blending near the seam to make
it less visible
4. Adjust the horizontal location of the seam (it can be moved further to the left as
the right image overlaps into the left by quite a few pixels)

Note that you do not have to try all of the above. However, you will only receive a mark here depending on
• the quality of the stitched image
• whether a serious programming attempt is made to improve the visual quality of the stitched image

In [ ]:
# Write your code here

# The right image appears darker: scale the brightness of the right image and then stitch the images together again
brightness_scale = 1.1
bright_img = np.clip(img_right * brightness_scale, 0, 255).astype(np.uint8)
bright_stitch_img = stitch_images(img_left, bright_img, homography)

# loops through each column looking for the first full non-zero column and crops the image at that point
# Output
# x - int: the value of the full non-zero column
def find_col(img):
     for x in range(img.shape[1] - 1, 0, -1):
          col = img[:, x]
          if (col != 0).all():
               return x

# loops through each row looking for the first full non-zero row and crops the image at that point
# Output
# y - int: the value of the full non-zero row
def find_row(img):
     for y in range(0, img.shape[0] - 1, 1):
          row = img[y, :img.shape[0] - 1]
          if (row != 0).all():
               return y

In [ ]:
# Write your code here
x_bound = find_col(bright_stitch_img)
y_bound = find_row(bright_stitch_img)

print('x bound is', x_bound)
print('y bound is', y_bound)

#array slicing to crop image by the above bounds
cropped_img = bright_stitch_img[y_bound:, :x_bound, :]

plt.figure(figsize=(45,15))
plt.subplot(1,2,1)
plt.imshow(cropped_img)

#apply gaussian kernel
blurred = cv2.GaussianBlur(cropped_img[0:440, 200:600], (25, 25), 10)

blur_crop = cropped_img
blur_crop[0:440, 200:600] = blurred

plt.subplot(1,2,2)
plt.imshow(blur_crop)
plt.show()

# Task 6: Now try your own!

In this final task, you will:
1. Take two images from different perspective of the same scenery and display it
2. Find and match key points across the two images
3. Calculate the homography matrix1 . Print out the homography matrix that you end
up using.
4. Apply image stitching and quality improvement for a final image (from tasks 1 to 5)

In [ ]:
# Write your code here


In [ ]:
# Show results here





# Discussion:

Write a brief (600 word max) report describing how you solved each task, interpreting the results and pointing to any insights gained along the way. For example, you may wish to explain what a homography is, the conditions under which it can be used. Analyse the stitched images and explain any interesting artifacts you may see and why these occur. You should discuss the reasons for the presence of the image seam, and the strategies you used to improve this, pointing to the theory you have learned in class that guided your solution. Finally, explain any changes you made to the code to apply it to your own images. 



To stitch the two initial images together the right image was warped and distorted in order to align the keypoints and stitch the two images together. This process whilst connecting the two images is what created the image seam with a large difference in the top left corner of the right image. 

To counteract this alteration and improve the consistency of the stitched image, we employed both a brightness modifier and a gaussian blur across the region in which the seam was visible. By adding on a brightness modifier to the right image we were attempting to counteract the distortion of brightness created by the warping of the image shape. A brightness factor of 1.1 was chosen to maintain cohesion across the rest of the image when stitched while allowing the seam to contrast the left image less. Gaussian blur was added afterwards in a contained area around the image seam to keep image quality in ther rest of the image high but still allow a strong blur to even out the remaining contrast left by the brightness scaling.

